# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. Unit of analysis + time window

**Unit of analysis:** one content page for one client.

For the eventual ranking task, each row represents a single content item and its observable search/content signals during a defined feature window.

The feature window must end before the outcome window begins. This separation is important because the eventual target is intended to represent a future deterioration in visibility, not the same-period movement already visible in the features.

For the starter dataset, the available performance fields are based on trailing windows, including 90-day impressions, clicks, and sessions. The starter `trend_direction` describes a current/recent movement and therefore will be treated as an exploratory label source rather than as a feature or final future target.

The exact future outcome window and threshold will be finalized after checking the available date/time fields and ensuring that the feature and target windows do not overlap.

In [11]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nDate/time-like columns:")
date_cols = [
    col for col in df.columns
    if any(x in col.lower() for x in ["date", "time", "day"])
]
print(date_cols)

Dataset shape: (30000, 44)
Unique content items: 30000
Unique clients: 32

Date/time-like columns:
['days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update']


## 2. Fields: feature / label / context / excluded

### Feature candidates

The initial feature set will contain page-level signals that would be available before the future outcome window, including:

* `content_age_days`
* `days_since_last_update`
* `impressions_90d`
* `clicks_90d`
* `sessions_90d`
* `avg_position`

Additional fields may be considered after checking their meaning, availability, missingness, and leakage risk.

### Label

The intended future-looking label is:

**`future_decline`** — whether the page experiences a predefined meaningful deterioration in search visibility during the future outcome window.

The exact threshold and window are not fixed yet because the available temporal structure must be verified first.

### Context

`client_id` and `content_id` are contextual identifiers. They are useful for grouping, auditing, and client-aware validation, but should not automatically be treated as predictive features.

### Excluded

`trend_direction` and `trend_pct` are excluded from the feature set because they are derived from recent impression movement and can directly encode the outcome being modeled.

Identifiers such as `content_id` are also excluded from model features because their numeric or categorical identity does not represent a meaningful content signal.

The final feature list will only include variables that are available before the target window and pass the later leakage checks.

In [12]:
# This cell is for CODE (numbers, a query, a check).
feature_candidates = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
]

context_fields = [
    "client_id",
    "content_id",
]

excluded_fields = [
    "trend_direction",
    "trend_pct",
]

print("FEATURE CANDIDATES")
for col in feature_candidates:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

print("\nCONTEXT")
for col in context_fields:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

print("\nEXCLUDED")
for col in excluded_fields:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

FEATURE CANDIDATES
content_age_days: FOUND
days_since_last_update: FOUND
impressions_90d: FOUND
clicks_90d: FOUND
sessions_90d: FOUND
avg_position: FOUND

CONTEXT
client_id: FOUND
content_id: FOUND

EXCLUDED
trend_direction: FOUND
trend_pct: FOUND


## 3. Verify it with queries

The contract should be backed by reproducible checks rather than assumptions.

I will verify that:

* each row corresponds to one content item;
* the dataset contains the expected number of content items and clients;
* the proposed feature fields exist;
* missing values are measurable;
* the available time/window fields can be identified;
* the starter trend fields are present but excluded from model features.

These checks establish what the current dataset can actually support before a final target window is designed.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# 1. Grain check

print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

# 2. Client count

print("\nUnique clients:", df["client_id"].nunique())

# 3. Missingness for proposed features

check_cols = feature_candidates + context_fields + excluded_fields

missing = (
    df[check_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
).round(2)

print("\nMissing values:")
display(missing)

# 4. Basic ranges

print("\nFeature ranges:")
display(
    df[feature_candidates]
    .describe()
    .T[["count", "min", "50%", "max"]]
)

Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

Unique clients: 32

Missing values:


,missing_count,missing_pct
trend_pct,3388,11.29
content_age_days,0,0.00
days_since_last_update,0,0.00
impressions_90d,0,0.00
sessions_90d,0,0.00
clicks_90d,0,0.00
avg_position,0,0.00
client_id,0,0.00
content_id,0,0.00
trend_direction,0,0.00



Feature ranges:


,count,min,50%,max
content_age_days,30000.0,90.0,236.0,564.0
days_since_last_update,30000.0,1.0,20.0,373.0
impressions_90d,30000.0,1.0,731.0,517715.0
clicks_90d,30000.0,0.0,1.0,4178.0
sessions_90d,30000.0,1.0,7.0,4345.0
avg_position,30000.0,0.0,10.8,245.0


## 4. Data limits

There are several limitations that affect the eventual ML task.

**First, the starter dataset does not necessarily provide a complete time history for every page.** Some pages may have shorter or less complete observation histories, so comparisons across pages may not be equally supported.

**Second, some early observations may contain GSC-only or otherwise incomplete performance information.** This can create differences in data availability that should be measured rather than assumed away.

**Third, window overlap is a major leakage risk.** The feature window and outcome window must be separated. A feature calculated using information from the same period as the target could make the model appear stronger without representing a valid future prediction.

**Fourth, the starter `trend_direction` is not a future outcome.** It describes recent movement in the available observation window, so it is useful for exploration but should not be treated as the final predictive target.

Finally, the dataset can support measured associations and decision-support experiments, but it cannot by itself establish that a particular content intervention caused a ranking or traffic improvement.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Check the distribution of the exploratory trend label.

print("Trend direction distribution:")
display(
    df["trend_direction"]
      .value_counts(dropna=False)
      .rename_axis("trend_direction")
      .to_frame("count")
)

print("\nTrend direction share:")
display(
    (
        df["trend_direction"]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
        .rename_axis("trend_direction")
        .to_frame("percentage")
    )
)

# Check whether any date/time columns were found.
print("\nPotential temporal columns:")
print(date_cols if date_cols else "No explicit date/time columns found.")

Trend direction distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Trend direction share:


,percentage
trend_direction,
down,54.21
stable,19.87
up,14.63
new,7.45
flat,3.84



Potential temporal columns:
['days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update']


## Self-check

* [x] Unit of analysis is explicitly defined as one content page for one client.
* [x] Candidate feature fields are explicitly listed.
* [x] The future-looking target is defined conceptually.
* [x] `trend_direction` and `trend_pct` are excluded from model features.
* [x] Client and content identifiers are treated as context rather than automatic features.
* [x] Grain is verified with actual dataframe checks.
* [x] Counts and missing values are measured.
* [x] Available temporal fields are checked rather than assumed.
* [x] Feature/outcome window overlap is identified as a leakage risk.
* [x] Dataset limitations are documented.
* [x] Claims use careful language: observed, measured, directional, and decision-support.
* [x] No private client information is included.
* [x] The notebook runs from top to bottom without errors.